In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import os
current_pwd = os.getcwd()

possible_paths = [
    '/home/export/soheuny/SRFinder/soheun/notebooks', 
    '/home/soheuny/HH4bsim/soheun/notebooks',
    '/home/export/soheuny/SRFinder/soheun/notebooks/draft'
]
    
assert os.getcwd() in possible_paths, f"Did you change the path? It should be one of {possible_paths}"
os.chdir("/home/export/soheuny/SRFinder/soheun")

In [2]:
from typing import Tuple, Any
from constants import FEATURES
from signal_region import compute_sr_stats, get_SR_CR_cut
from events_data import events_from_scdinfo
from dataset import MotherSamples
from training_info import TrainingInfo

SIGNAL_FILENAME = "HH4b_picoAOD.h5"

def load_events_data(hash_: str) -> Tuple[Any, Any]:
    """Load training and test events data from a hash."""
    CR_fvt_tinfo = TrainingInfo.load(hash_)
    SR_stats_hashes = CR_fvt_tinfo.hparams["signal_region"]["SR_stats_hashes"]
    smeared_tinfo = TrainingInfo.load(SR_stats_hashes[0])
    msamples = MotherSamples.load(smeared_tinfo.ms_hash)
    
    events_train = events_from_scdinfo(
        msamples.scdinfo[smeared_tinfo.ms_idx], 
        FEATURES, 
        SIGNAL_FILENAME
    )
    events_tst = events_from_scdinfo(
        msamples.scdinfo[~smeared_tinfo.ms_idx], 
        FEATURES, 
        SIGNAL_FILENAME
    )
    return events_train, events_tst


In [3]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from plots import hist_events_by_labels
from events_data import EventsData
from fvt_classifier import FvTClassifier
# import LogNorm
from matplotlib.colors import LogNorm
from training_info import TrainingInfo
from plots import plot_rewighted_samples_by_model, plot_samples_raw
from dataset import MotherSamples
from events_data import events_from_scdinfo
import pickle

features = [
    "sym_Jet0_pt", "sym_Jet1_pt", "sym_Jet2_pt", "sym_Jet3_pt",
    "sym_Jet0_eta", "sym_Jet1_eta", "sym_Jet2_eta", "sym_Jet3_eta",
    "sym_Jet0_phi", "sym_Jet1_phi", "sym_Jet2_phi", "sym_Jet3_phi",  
    "sym_Jet0_m", "sym_Jet1_m", "sym_Jet2_m", "sym_Jet3_m",
]

# use tex
plt.rcParams["text.usetex"] = True
# plt.rcParams["font.family"] = "serif"
# plt.rcParams["font.serif"] = "Times New Roman"

plt.rcParams["figure.dpi"] = 100
plt.rcParams["figure.titlesize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["axes.labelsize"] = 15
plt.rcParams["xtick.labelsize"] = 15
plt.rcParams["ytick.labelsize"] = 15
plt.rcParams["figure.labelsize"] = 20
plt.rcParams["lines.markersize"] = 3
# legend font size
plt.rcParams["legend.fontsize"] = 15

import pandas as pd

path_3b = Path("../events/MG3/dataframes/threeTag_picoAOD.h5")
path_4b = Path("../events/MG3/dataframes/fourTag_10x_picoAOD.h5")
path_signal = Path("../events/MG3/dataframes/HH4b_picoAOD.h5")
df_3b = pd.read_hdf(path_3b)
df_bg4b = pd.read_hdf(path_4b)
df_signal = pd.read_hdf(path_signal)
df_3b["signal"] = False
df_bg4b["signal"] = False
df_signal["signal"] = True
raw_df_list = [df_3b, df_bg4b, df_signal]
loaded_df = {path_3b: df_3b, path_4b: df_bg4b, path_signal: df_signal}

In [ ]:
n_reps = 1000
cdf_mode = "mean"
storage = f"affine_correction_and_ks_poisson_bootstrap_n_reps={n_reps}_cdf_mode={cdf_mode}"

from tqdm import tqdm

n_reps = 1000
sig_level = 0.05
SR_size = 0.2
experiment_name = "CR_fvt_training_ensemble_max"

hashes, hparams = TrainingInfo.find({"experiment_name": experiment_name, 
                                    #  "signal_region": lambda x: x["4b_in_SR"] == SR_size
                                     }, 
                                     return_hparams=True)
metadata = TrainingInfo.load_metadata()
seeds = [metadata[hash_]["dataset"]["seed"] for hash_ in hashes]
hashes = [hashes[i] for i in np.argsort(seeds)]

results = []

np.random.seed(0)
for hash_ in tqdm(hashes):
    CR_fvt_tinfo = TrainingInfo.load(hash_)
    seed = CR_fvt_tinfo.hparams["dataset"]["seed"]
    signal_ratio = CR_fvt_tinfo.hparams["dataset"]["signal_ratio"]
    SR_size = CR_fvt_tinfo.hparams["signal_region"]["4b_in_SR"]
    SR_stats_hashes = CR_fvt_tinfo.hparams["signal_region"]["SR_stats_hashes"]
    
    smeared_tinfo = TrainingInfo.load(SR_stats_hashes[0])
    if CR_fvt_tinfo.hparams["signal_region"]["stats_type"] == "smeared":
        noise_scale = smeared_tinfo.hparams["smearing"]["noise_scale"]
    else:
        noise_scale = np.inf
        
    
    correction_results = CR_fvt_tinfo.aux_info[storage]
    
    alt_value_correction = correction_results["alt_value_correction"]
    null_values_correction = correction_results["null_values_correction"]
    alt_value_no_correction = correction_results["alt_value_no_correction"]
    null_values_no_correction = correction_results["null_values_no_correction"]
    p_value_correction = correction_results["p_value_correction"]
    p_value_no_correction = correction_results["p_value_no_correction"]
    
    results.append({
        "seed": seed,
        "signal_ratio": signal_ratio,
        "SR_size": SR_size,
        "noise_scale": noise_scale,
        "p_value_correction": p_value_correction,
        "p_value_no_correction": p_value_no_correction,
        "alt_value_correction": alt_value_correction,
        "null_values_correction": null_values_correction,
        "alt_value_no_correction": alt_value_no_correction,
        "null_values_no_correction": null_values_no_correction,
        "correction_slope": correction_results["correction_slope"],
        "correction_intercept": correction_results["correction_intercept"],
        "stats_3b_mean": correction_results["stats_3b_mean"],
        "stats_3b_std": correction_results["stats_3b_std"],
    })

results_df = pd.DataFrame(results)

results_df["reject_null_correction"] = results_df["p_value_correction"] <= sig_level
results_df["reject_null_no_correction"] = results_df["p_value_no_correction"] <= sig_level
pd.set_option("display.max_rows", 100)
for signal_ratio in [0.0, 0.005, 0.0075, 0.01, 0.02]:
    print(f"signal_ratio = {signal_ratio}")
    tmp_df = results_df[results_df["signal_ratio"] == signal_ratio]
    tmp_df = tmp_df.groupby(["SR_size", "noise_scale"]).agg({
        "reject_null_correction": "mean",
        "reject_null_no_correction": "mean"
    })
    display(tmp_df)

 33%|███▎      | 1667/5000 [01:52<03:44, 14.82it/s]


KeyboardInterrupt: 

In [4]:
n_reps = 1000
cdf_mode = "mean"
storage = f"affine_correction_and_ks_poisson_bootstrap_n_reps={n_reps}_cdf_mode={cdf_mode}"

from tqdm import tqdm

n_reps = 1000
sig_level = 0.05
SR_size = 0.2
experiment_name = "CR_fvt_training_ensemble_max"

hashes, hparams = TrainingInfo.find({"experiment_name": experiment_name, 
                                    #  "signal_region": lambda x: x["4b_in_SR"] == SR_size
                                     }, 
                                     return_hparams=True)
metadata = TrainingInfo.load_metadata()
seeds = [metadata[hash_]["dataset"]["seed"] for hash_ in hashes]
hashes = [hashes[i] for i in np.argsort(seeds)]

results = []

np.random.seed(0)
for hash_ in tqdm(hashes):
    CR_fvt_tinfo = TrainingInfo.load(hash_)
    seed = CR_fvt_tinfo.hparams["dataset"]["seed"]
    signal_ratio = CR_fvt_tinfo.hparams["dataset"]["signal_ratio"]
    SR_size = CR_fvt_tinfo.hparams["signal_region"]["4b_in_SR"]
    SR_stats_hashes = CR_fvt_tinfo.hparams["signal_region"]["SR_stats_hashes"]
    
    smeared_tinfo = TrainingInfo.load(SR_stats_hashes[0])
    if CR_fvt_tinfo.hparams["signal_region"]["stats_type"] == "smeared":
        noise_scale = smeared_tinfo.hparams["smearing"]["noise_scale"]
    else:
        noise_scale = np.inf
    
    signal_filename = CR_fvt_tinfo.hparams["dataset"]["signal_filename"]
    ensemble_mode = CR_fvt_tinfo.hparams["signal_region"]["ensemble_mode"]
    stats_type = CR_fvt_tinfo.hparams["signal_region"]["stats_type"]
        
    correction_results = CR_fvt_tinfo.aux_info[storage]
    
    alt_value_correction = correction_results["alt_value_correction"]
    null_values_correction = correction_results["null_values_correction"]
    alt_value_no_correction = correction_results["alt_value_no_correction"]
    null_values_no_correction = correction_results["null_values_no_correction"]
    p_value_correction = correction_results["p_value_correction"]
    p_value_no_correction = correction_results["p_value_no_correction"]
    
    results.append({
        "seed": seed,
        "signal_ratio": signal_ratio,
        "SR_size": SR_size,
        "noise_scale": noise_scale,
        "p_value_correction": p_value_correction,
        "p_value_no_correction": p_value_no_correction,
        "alt_value_correction": alt_value_correction,
        "null_values_correction": null_values_correction,
        "alt_value_no_correction": alt_value_no_correction,
        "null_values_no_correction": null_values_no_correction,
        "correction_slope": correction_results["correction_slope"],
        "correction_intercept": correction_results["correction_intercept"],
        "stats_3b_mean": correction_results["stats_3b_mean"],
        "stats_3b_std": correction_results["stats_3b_std"],
        # "stats_3b_min": np.min(stats_3b),
        # "stats_3b_max": np.max(stats_3b),
    })

results_df = pd.DataFrame(results)

100%|██████████| 5000/5000 [05:06<00:00, 16.30it/s]


In [6]:
def plot_mean_and_fill_between_std(x, y_list, ax, **kwargs):
    y_mean = np.mean(y_list, axis=0)
    y_std = np.std(y_list, axis=0)
    ax.plot(x, y_mean, **kwargs)
    if "color" in kwargs:
        ax.fill_between(x, y_mean - y_std, y_mean + y_std, alpha=0.2, color=kwargs["color"])
    else:
        ax.fill_between(x, y_mean - y_std, y_mean + y_std, alpha=0.2)
        
def get_events_tst(smeared_hash: str):
    smeared_fvt_tinfo = TrainingInfo.load(smeared_hash)
        
    base_encoder_hash = smeared_fvt_tinfo.hparams["encoder_hash"]
    base_fvt_tinfo = TrainingInfo.load(base_encoder_hash)
    ms_hash = base_fvt_tinfo.ms_hash
    ms_idx = base_fvt_tinfo.ms_idx
    msamples = MotherSamples.load(ms_hash)
    tst_scdinfo = msamples.scdinfo[~ms_idx]
    df_tst = tst_scdinfo.fetch_data(loaded_df)
    events_tst = EventsData.from_dataframe(df_tst, features)
    return events_tst


# HH4b Resonant 400

In [8]:
from tqdm import tqdm

n_reps = 1000
cdf_mode = "mean"
storage = f"affine_correction_and_ks_poisson_bootstrap_n_reps={n_reps}_cdf_mode={cdf_mode}"
sig_level = 0.05

for signal_name in [
    "HH4b", "HH4b_400", "HH4b_800"]:

    print("Signal name: ", signal_name)
    if signal_name == "HH4b":
        experiment_name = "CR_fvt_training_ensemble_max"
    elif signal_name == "HH4b_400":
        experiment_name = "CR_fvt_training_ensemble_max_HH4b_400"
    elif signal_name == "HH4b_800":
        experiment_name = "CR_fvt_training_ensemble_max_HH4b_800"

    hashes, hparams = TrainingInfo.find({"experiment_name": experiment_name}, 
                                        return_hparams=True)
    metadata = TrainingInfo.load_metadata()
    seeds = [metadata[hash_]["dataset"]["seed"] for hash_ in hashes]
    hashes = [hashes[i] for i in np.argsort(seeds)]

    results = []

    np.random.seed(0)
    for hash_ in tqdm(hashes):
        CR_fvt_tinfo = TrainingInfo.load(hash_)
        seed = CR_fvt_tinfo.hparams["dataset"]["seed"]
        signal_ratio = CR_fvt_tinfo.hparams["dataset"]["signal_ratio"]
        SR_size = CR_fvt_tinfo.hparams["signal_region"]["4b_in_SR"]
        SR_stats_hashes = CR_fvt_tinfo.hparams["signal_region"]["SR_stats_hashes"]
        
        smeared_tinfo = TrainingInfo.load(SR_stats_hashes[0])
        if CR_fvt_tinfo.hparams["signal_region"]["stats_type"] == "smeared":
            noise_scale = smeared_tinfo.hparams["smearing"]["noise_scale"]
        else:
            noise_scale = np.inf
            
        
        correction_results = CR_fvt_tinfo.aux_info[storage]
        
        alt_value_correction = correction_results["alt_value_correction"]
        null_values_correction = correction_results["null_values_correction"]
        alt_value_no_correction = correction_results["alt_value_no_correction"]
        null_values_no_correction = correction_results["null_values_no_correction"]
        p_value_correction = correction_results["p_value_correction"]
        p_value_no_correction = correction_results["p_value_no_correction"]
        
        results.append({
            "seed": seed,
            "signal_ratio": signal_ratio,
            "SR_size": SR_size,
            "noise_scale": noise_scale,
            "p_value_correction": p_value_correction,
            "p_value_no_correction": p_value_no_correction,
            "alt_value_correction": alt_value_correction,
            "null_values_correction": null_values_correction,
            "alt_value_no_correction": alt_value_no_correction,
            "null_values_no_correction": null_values_no_correction,
            "correction_slope": correction_results["correction_slope"],
            "correction_intercept": correction_results["correction_intercept"],
            "stats_3b_mean": correction_results["stats_3b_mean"],
            "stats_3b_std": correction_results["stats_3b_std"],
        })

    results_df = pd.DataFrame(results)

    results_df["reject_null_correction"] = results_df["p_value_correction"] <= sig_level
    results_df["reject_null_no_correction"] = results_df["p_value_no_correction"] <= sig_level
    pd.set_option("display.max_rows", 100)
    for signal_ratio in [0.0, 0.005, 0.0075, 0.01, 0.02]:
        print(f"signal_ratio = {signal_ratio}")
        tmp_df = results_df[results_df["signal_ratio"] == signal_ratio]
        tmp_df = tmp_df.groupby(["SR_size", "noise_scale"]).agg({
            "reject_null_correction": "mean",
            "reject_null_no_correction": "mean"
        })
        display(tmp_df)
    
    for noise_scale in [0.5, 1.0, 2.0, 3.0, np.inf]:
        tmp_df = results_df[results_df["noise_scale"] == noise_scale]
        tmp_df = tmp_df.groupby(["SR_size", "signal_ratio"]).agg({
            "reject_null_correction": "mean",
        })
        # tmp_df to pivot table
        tmp_df = tmp_df.reset_index().pivot(index="SR_size", columns="signal_ratio", values=["reject_null_correction"])
        tmp_df.to_csv(f"./notebooks/draft/csv/hypothesis_testing_{signal_name}_noise_scale={noise_scale}.csv")

Signal name:  HH4b


100%|██████████| 5000/5000 [00:17<00:00, 288.21it/s]


signal_ratio = 0.0


reject_null_correction  reject_null_no_correction
SR_size noise_scale                                                   
0.05    0.5                            0.06                       0.24
        1.0                            0.04                       0.28
        2.0                            0.04                       0.18
        3.0                            0.02                       0.18
        inf                            0.00                       0.22
0.10    0.5                            0.06                       0.40
        1.0                            0.06                       0.36
        2.0                            0.00                       0.18
        3.0                            0.02                       0.40
        inf                            0.02                       0.48
0.15    0.5                            0.06                       0.48
        1.0                            0.02                       0.40
        2.0                            0.00                       0.30
        3.0                            0.02                       0.46
        inf                            0.04                       0.52
0.20    0.5                            0.08                       0.56
        1.0                            0.02                       0.54
        2.0                            0.02                       0.34
        3.0                            0.02                       0.66
        inf                            0.00                       0.64

signal_ratio = 0.005


...

reject_null_correction  reject_null_no_correction
SR_size noise_scale                                                   
0.05    0.5                            0.98                        1.0
        1.0                            1.00                        1.0
        2.0                            1.00                        1.0
        3.0                            1.00                        1.0
        inf                            1.00                        1.0
0.10    0.5                            1.00                        1.0
        1.0                            1.00                        1.0
        2.0                            1.00                        1.0
        3.0                            1.00                        1.0
        inf                            1.00                        1.0
0.15    0.5                            1.00                        1.0
        1.0                            1.00                        1.0
        2.0                            1.00                        1.0
        3.0                            1.00                        1.0
        inf                            1.00                        1.0
0.20    0.5                            1.00                        1.0
        1.0                            1.00                        1.0
        2.0                            1.00                        1.0
        3.0                            1.00                        1.0
        inf                            1.00                        1.0

signal_ratio = 0.01


reject_null_correction  reject_null_no_correction
SR_size noise_scale                                                   
0.05    0.5                            1.00                        1.0
        1.0                            0.98                        1.0
        2.0                            1.00                        1.0
        3.0                            1.00                        1.0
        inf                            1.00                        1.0
0.10    0.5                            1.00                        1.0
        1.0                            1.00                        1.0
        2.0                            1.00                        1.0
        3.0                            1.00                        1.0
        inf                            1.00                        1.0
0.15    0.5                            1.00                        1.0
        1.0                            0.98                        1.0
        2.0                            1.00                        1.0
        3.0                            1.00                        1.0
        inf                            1.00                        1.0
0.20    0.5                            1.00                        1.0
        1.0                            0.98                        1.0
        2.0                            1.00                        1.0
        3.0                            1.00                        1.0
        inf                            1.00                        1.0

signal_ratio = 0.02


reject_null_correction  reject_null_no_correction
SR_size noise_scale                                                   
0.05    0.5                            1.00                       1.00
        1.0                            0.96                       0.98
        2.0                            1.00                       1.00
        3.0                            0.98                       0.98
        inf                            1.00                       1.00
0.10    0.5                            1.00                       1.00
        1.0                            1.00                       1.00
        2.0                            1.00                       1.00
        3.0                            1.00                       1.00
        inf                            1.00                       1.00
0.15    0.5                            1.00                       1.00
        1.0                            1.00                       1.00
        2.0                            1.00                       1.00
        3.0                            1.00                       1.00
        inf                            1.00                       1.00
0.20    0.5                            1.00                       1.00
        1.0                            1.00                       1.00
        2.0                            1.00                       1.00
        3.0                            1.00                       1.00
        inf                            1.00                       1.00